# EP17. Semantic Caching과 Prompt Compression

## 학습 목표

1. **Exact Cache와 Semantic Cache**의 차이를 이해하고 각각 구현한다
2. **임베딩 유사도 기반** Semantic Cache를 구축하고 유사도 임계값을 튜닝한다
3. **Claude Prompt Caching**(cache_control)의 원리와 적용법을 익힌다
4. **EP07 Context Compression과 결합**하여 비용 절감 효과를 극대화한다

## AI 가이드

> 막히는 부분은 Claude에게 물어보세요.
> 예시 프롬프트:
> - "Semantic Cache에서 유사도 임계값은 어떻게 결정하나요?"
> - "Redis 벡터 검색과 ChromaDB 유사도 검색의 차이는?"
> - "Prompt Caching의 cache_control 설정 방법을 알려주세요"

## 사전 요구사항

- Python 3.10+
- Anthropic API 키 (Claude 호출용)
- (선택) Langfuse 계정 (모니터링용)
- EP07 Context Compression 수강 권장

| 항목 | 내용 |
|------|------|
| 예상 소요 시간 | 60분 |
| 난이도 | ⭐⭐ |
| API 비용 | ~$0.50 (실습 전체) |
| 필수 API 키 | `ANTHROPIC_API_KEY` |

---
## 섹션 1. 환경 설정

In [ ]:
# 필요한 패키지 설치
!uv pip install redis fakeredis langchain langchain-anthropic langfuse chromadb sentence-transformers tiktoken python-dotenv --quiet

---
## 섹션 2. 라이브러리 로드 및 API 키 설정

In [ ]:
import os
import json
import time
import hashlib
import numpy as np
from typing import Optional
from dotenv import load_dotenv

load_dotenv()

# API 키 설정 (환경변수 또는 직접 입력)
os.environ.setdefault("ANTHROPIC_API_KEY", "your-anthropic-api-key-here")
os.environ.setdefault("LANGFUSE_PUBLIC_KEY", "your-langfuse-public-key")
os.environ.setdefault("LANGFUSE_SECRET_KEY", "your-langfuse-secret-key")
os.environ.setdefault("LANGFUSE_HOST", "https://cloud.langfuse.com")

print("라이브러리 로드 완료")
print(f"ANTHROPIC_API_KEY 설정: {'✓' if os.getenv('ANTHROPIC_API_KEY') != 'your-anthropic-api-key-here' else '✗ (설정 필요)'}")

In [ ]:
# Redis 연결 (fakeredis 폴백)
try:
    import redis
    r = redis.Redis()
    r.ping()
    print("Redis 서버 연결 성공")
    REDIS_AVAILABLE = True
except Exception:
    import fakeredis
    r = fakeredis.FakeRedis()
    print("Redis 서버 미발견 → fakeredis 사용 (로컬 테스트 모드)")
    REDIS_AVAILABLE = False

# 임베딩 모델 로드
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"임베딩 모델 로드 완료 (차원: {embedding_model.get_sentence_embedding_dimension()})")

---
## 섹션 3. Exact Cache 구현

가장 단순한 캐싱 전략: 쿼리 문자열의 **해시값을 키**로 사용하여 완전히 동일한 질문만 캐시 히트합니다.

In [ ]:
class ExactCache:
    """문자열 완전 일치 기반 Exact Cache"""
    
    def __init__(self):
        self.cache = {}  # {hash: {"query": str, "response": str, "timestamp": float}}
        self.stats = {"hits": 0, "misses": 0}
    
    def _hash_query(self, query: str) -> str:
        """쿼리 문자열을 SHA-256 해시로 변환"""
        normalized = query.strip().lower()
        return hashlib.sha256(normalized.encode()).hexdigest()
    
    def get(self, query: str) -> Optional[str]:
        """캐시 조회 -- 완전 일치 시에만 반환"""
        key = self._hash_query(query)
        if key in self.cache:
            self.stats["hits"] += 1
            return self.cache[key]["response"]
        self.stats["misses"] += 1
        return None
    
    def put(self, query: str, response: str):
        """캐시에 쿼리-응답 쌍 저장"""
        key = self._hash_query(query)
        self.cache[key] = {
            "query": query,
            "response": response,
            "timestamp": time.time()
        }
    
    def hit_rate(self) -> float:
        total = self.stats["hits"] + self.stats["misses"]
        return self.stats["hits"] / total * 100 if total > 0 else 0.0
    
    def __repr__(self):
        return (f"ExactCache(entries={len(self.cache)}, "
                f"hits={self.stats['hits']}, misses={self.stats['misses']}, "
                f"hit_rate={self.hit_rate():.1f}%)")


# Exact Cache 테스트
exact_cache = ExactCache()

# 캐시에 항목 저장
exact_cache.put("환불 절차가 어떻게 되나요?", "환불은 주문 후 7일 이내에 가능합니다...")
exact_cache.put("배송 기간은 얼마나 걸리나요?", "일반 배송은 2~3일, 특급 배송은 당일 도착합니다.")

# 테스트 쿼리
test_queries = [
    "환불 절차가 어떻게 되나요?",       # 완전 일치 → 히트
    "환불하려면 어떻게 해야 하나요?",     # 의미 유사, 문자열 다름 → 미스
    "반품 방법 알려주세요",              # 의미 유사, 문자열 다름 → 미스
    "배송 기간은 얼마나 걸리나요?",       # 완전 일치 → 히트
]

print("=" * 60)
print("Exact Cache 테스트")
print("=" * 60)
for q in test_queries:
    result = exact_cache.get(q)
    status = "HIT" if result else "MISS"
    print(f"[{status}] {q}")
    if result:
        print(f"       → {result[:50]}...")

print(f"\n{exact_cache}")
print(f"\n한계: 의미가 같아도 문자열이 다르면 캐시 미스 발생!")

---
## 섹션 4. Semantic Cache 구현

Exact Cache의 한계를 극복하기 위해, **임베딩 벡터의 코사인 유사도**를 사용하여 의미적으로 유사한 질문도 캐시 히트로 처리합니다.

In [ ]:
class SemanticCache:
    """임베딩 유사도 기반 Semantic Cache
    
    저장: 쿼리 임베딩 + 응답을 Redis(또는 fakeredis)에 저장
    조회: 새 쿼리의 임베딩과 캐시된 임베딩의 코사인 유사도로 매칭
    """
    
    def __init__(self, embedding_model, redis_client, threshold: float = 0.92):
        self.embedding_model = embedding_model
        self.redis_client = redis_client
        self.threshold = threshold
        self.entries = []  # [{"query": str, "embedding": ndarray, "response": str, "timestamp": float}]
        self.stats = {"hits": 0, "misses": 0, "similarities": []}
    
    def _get_embedding(self, text: str) -> np.ndarray:
        """텍스트를 임베딩 벡터로 변환"""
        return self.embedding_model.encode(text, normalize_embeddings=True)
    
    def _cosine_similarity(self, a: np.ndarray, b: np.ndarray) -> float:
        """코사인 유사도 계산 (정규화된 벡터: 내적 = 코사인 유사도)"""
        return float(np.dot(a, b))
    
    def get(self, query: str) -> Optional[dict]:
        """Semantic Cache 조회
        
        Returns:
            dict with keys: response, similarity, cached_query
            None if cache miss
        """
        query_emb = self._get_embedding(query)
        
        best_match = None
        best_similarity = -1.0
        
        for entry in self.entries:
            sim = self._cosine_similarity(query_emb, entry["embedding"])
            if sim > best_similarity:
                best_similarity = sim
                best_match = entry
        
        self.stats["similarities"].append(best_similarity)
        
        if best_match and best_similarity >= self.threshold:
            self.stats["hits"] += 1
            return {
                "response": best_match["response"],
                "similarity": best_similarity,
                "cached_query": best_match["query"]
            }
        
        self.stats["misses"] += 1
        return None
    
    def put(self, query: str, response: str):
        """Semantic Cache에 항목 저장"""
        embedding = self._get_embedding(query)
        entry = {
            "query": query,
            "embedding": embedding,
            "response": response,
            "timestamp": time.time()
        }
        self.entries.append(entry)
        
        # Redis에도 저장 (메타데이터용)
        cache_key = f"sem_cache:{hashlib.md5(query.encode()).hexdigest()}"
        self.redis_client.setex(
            cache_key,
            3600,  # TTL: 1시간
            json.dumps({"query": query, "response": response, "timestamp": time.time()})
        )
    
    def hit_rate(self) -> float:
        total = self.stats["hits"] + self.stats["misses"]
        return self.stats["hits"] / total * 100 if total > 0 else 0.0
    
    def __repr__(self):
        return (f"SemanticCache(entries={len(self.entries)}, threshold={self.threshold}, "
                f"hits={self.stats['hits']}, misses={self.stats['misses']}, "
                f"hit_rate={self.hit_rate():.1f}%)")

In [ ]:
# Semantic Cache 테스트
sem_cache = SemanticCache(
    embedding_model=embedding_model,
    redis_client=r,
    threshold=0.92
)

# 캐시에 FAQ 항목 저장
faq_data = [
    ("환불 절차가 어떻게 되나요?", "환불은 주문 후 7일 이내에 가능합니다. 마이페이지 > 주문내역에서 환불 신청을 하시면 됩니다."),
    ("배송 기간은 얼마나 걸리나요?", "일반 배송은 2~3일, 특급 배송은 당일 도착합니다. 도서산간 지역은 1~2일 추가됩니다."),
    ("회원 탈퇴는 어떻게 하나요?", "설정 > 계정관리 > 회원탈퇴에서 진행 가능합니다. 탈퇴 후 30일간 복구 가능합니다."),
    ("포인트 적립은 언제 되나요?", "구매 확정 후 2~3일 이내에 자동 적립됩니다. 리뷰 작성 시 추가 포인트가 지급됩니다."),
]

for query, response in faq_data:
    sem_cache.put(query, response)

print(f"캐시에 {len(faq_data)}개 FAQ 항목 저장 완료\n")

# 유사 질문으로 테스트
test_queries = [
    "환불하려면 어떻게 해야 하나요?",          # 유사 → 히트 예상
    "반품 방법 알려주세요",                    # 유사 → 히트 예상
    "배송은 며칠 걸려요?",                     # 유사 → 히트 예상
    "회원 탈퇴하고 싶습니다",                  # 유사 → 히트 예상
    "오늘 날씨가 어때요?",                     # 관련 없음 → 미스 예상
    "주문을 취소하고 싶어요",                  # 부분 유사 → 경계 사례
]

print("=" * 70)
print("Semantic Cache 테스트 (threshold=0.92)")
print("=" * 70)

for q in test_queries:
    result = sem_cache.get(q)
    if result:
        print(f"[HIT]  {q}")
        print(f"       유사도: {result['similarity']:.4f}")
        print(f"       매칭된 캐시 질문: {result['cached_query']}")
        print(f"       응답: {result['response'][:60]}...")
    else:
        print(f"[MISS] {q}")
        best_sim = sem_cache.stats['similarities'][-1]
        print(f"       최고 유사도: {best_sim:.4f} (임계값 {sem_cache.threshold} 미달)")
    print()

print(sem_cache)

---
## 섹션 5. 유사도 임계값 실험

임계값(threshold)에 따라 **히트율**과 **정확도(올바른 캐시 반환)**가 달라집니다. 너무 낮으면 오답이 증가하고, 너무 높으면 히트율이 떨어집니다.

In [ ]:
# 유사도 임계값별 성능 비교 실험

# 테스트 데이터: (질문, 기대 매칭 FAQ 인덱스, 정답 여부가 맞는지)
# FAQ 인덱스: 0=환불, 1=배송, 2=탈퇴, 3=포인트, -1=매칭 없어야 함
test_data = [
    ("환불하려면 어떻게 해야 하나요?", 0, True),
    ("반품 절차 알려주세요", 0, True),
    ("환불 가능한가요?", 0, True),
    ("배송은 며칠이나 걸려요?", 1, True),
    ("택배 언제 도착해요?", 1, True),
    ("회원 탈퇴하고 싶어요", 2, True),
    ("계정 삭제 방법", 2, True),
    ("포인트는 언제 들어오나요?", 3, True),
    ("적립금 확인하고 싶어요", 3, True),
    ("오늘 날씨 어때요?", -1, True),       # 미스가 정답
    ("맛있는 식당 추천해줘", -1, True),     # 미스가 정답
    ("주문을 취소하고 싶어요", -1, True),   # 환불과 유사하지만 다른 의도
]

thresholds = [0.85, 0.88, 0.90, 0.92, 0.95]
results = []

for threshold in thresholds:
    cache = SemanticCache(
        embedding_model=embedding_model,
        redis_client=r,
        threshold=threshold
    )
    # FAQ 데이터 동일하게 로드
    for query, response in faq_data:
        cache.put(query, response)
    
    hits = 0
    correct_hits = 0
    total = len(test_data)
    false_positives = 0
    
    for query, expected_idx, _ in test_data:
        result = cache.get(query)
        if result:
            hits += 1
            # 매칭된 캐시 질문이 기대한 FAQ와 일치하는지 확인
            matched_faq = result["cached_query"]
            actual_idx = next(
                (i for i, (q, _) in enumerate(faq_data) if q == matched_faq), -1
            )
            if actual_idx == expected_idx:
                correct_hits += 1
            elif expected_idx == -1:  # 미스가 정답인데 히트가 나온 경우
                false_positives += 1
        else:
            if expected_idx == -1:  # 미스가 정답인 경우
                correct_hits += 1
    
    hit_rate = hits / total * 100
    accuracy = correct_hits / total * 100
    fp_rate = false_positives / total * 100
    
    results.append({
        "threshold": threshold,
        "hit_rate": hit_rate,
        "accuracy": accuracy,
        "false_positive_rate": fp_rate
    })

# 결과 출력
print("=" * 70)
print("유사도 임계값별 Semantic Cache 성능 비교")
print("=" * 70)
print(f"{'임계값':>8} {'히트율':>10} {'정확도':>10} {'오답률':>10}")
print("-" * 45)
for r_item in results:
    marker = " ← 권장" if r_item["threshold"] == 0.92 else ""
    print(f"{r_item['threshold']:>8.2f} {r_item['hit_rate']:>9.1f}% {r_item['accuracy']:>9.1f}% {r_item['false_positive_rate']:>9.1f}%{marker}")

print("\n인사이트:")
print("  - 임계값이 낮을수록 히트율은 높지만 오답 위험 증가")
print("  - 임계값 0.92가 히트율과 정확도의 균형점")
print("  - 도메인 특성에 따라 A/B 테스트로 최적값을 찾아야 함")

---
## 섹션 6. Prompt Caching -- Claude의 네이티브 KV 캐시

Semantic Cache가 **응답 전체를 캐싱**하는 반면, Prompt Caching은 **LLM 내부의 KV(Key-Value) 연산 결과를 재사용**하여 비용을 절감합니다.

- Claude의 `cache_control` 기능을 사용
- 시스템 프롬프트, 긴 컨텍스트 등 **반복되는 prefix 부분**을 캐싱
- 캐시 히트 시 해당 부분의 입력 토큰 비용이 **90% 할인**

In [ ]:
# Claude Prompt Caching (cache_control) 데모

# 긴 시스템 프롬프트 (캐싱 대상)
SYSTEM_PROMPT = """당신은 전자상거래 고객 상담 AI입니다.

## 환불 정책
- 주문 후 7일 이내 환불 가능
- 사용/개봉 상품은 환불 불가
- 환불 처리 기간: 영업일 기준 3~5일
- 카드 결제 취소는 카드사에 따라 1~2주 소요

## 배송 정보
- 일반 배송: 2~3일 (무료)
- 특급 배송: 당일 (3,000원)
- 도서산간: 추가 1~2일
- 해외 배송: 7~14일

## 포인트/적립금
- 구매 확정 후 2~3일 이내 적립
- 리뷰 작성: 500포인트
- 포인트 유효기간: 1년
- 1포인트 = 1원

## 회원 관리
- 회원 탈퇴: 설정 > 계정관리
- 탈퇴 후 30일 복구 가능
- 개인정보 파기: 탈퇴 후 30일

항상 친절하고 정확한 정보를 제공하세요.
모르는 내용은 솔직히 "확인 후 안내드리겠습니다"라고 말하세요.
"""

print(f"시스템 프롬프트 길이: {len(SYSTEM_PROMPT)} 문자")
print(f"\nPrompt Caching 없이: 매 호출마다 전체 시스템 프롬프트 처리")
print(f"Prompt Caching 적용: 시스템 프롬프트를 캐시하여 비용 90% 절감")

In [ ]:
# Claude Prompt Caching 구현 예시
# (실제 API 호출 -- ANTHROPIC_API_KEY가 설정된 경우에만 실행)

def call_claude_with_prompt_caching(query: str, use_cache: bool = True):
    """Claude API 호출 (Prompt Caching 옵션)"""
    from anthropic import Anthropic
    client = Anthropic()
    
    system_content = {
        "type": "text",
        "text": SYSTEM_PROMPT,
    }
    
    # cache_control 추가 (캐싱 활성화 시)
    if use_cache:
        system_content["cache_control"] = {"type": "ephemeral"}
    
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=512,
        system=[system_content],
        messages=[{"role": "user", "content": query}]
    )
    
    # 캐시 사용 정보 확인
    usage = response.usage
    cache_info = {
        "input_tokens": usage.input_tokens,
        "output_tokens": usage.output_tokens,
        "cache_creation_input_tokens": getattr(usage, "cache_creation_input_tokens", 0),
        "cache_read_input_tokens": getattr(usage, "cache_read_input_tokens", 0),
    }
    
    return response.content[0].text, cache_info


# Prompt Caching 시연 (API 키가 있을 때만 실행)
if os.getenv("ANTHROPIC_API_KEY") != "your-anthropic-api-key-here":
    print("Claude Prompt Caching 시연\n")
    
    queries = ["환불 절차 알려주세요", "배송 기간이 궁금합니다", "포인트 적립 언제 되나요?"]
    
    for i, q in enumerate(queries):
        response_text, cache_info = call_claude_with_prompt_caching(q)
        print(f"--- 호출 {i+1}: {q} ---")
        print(f"응답: {response_text[:100]}...")
        print(f"입력 토큰: {cache_info['input_tokens']}")
        print(f"캐시 생성 토큰: {cache_info['cache_creation_input_tokens']}")
        print(f"캐시 읽기 토큰: {cache_info['cache_read_input_tokens']}")
        print()
    
    print("인사이트: 두 번째 호출부터 cache_read_input_tokens가 증가하면 캐시 히트!")
else:
    print("ANTHROPIC_API_KEY가 설정되지 않았습니다.")
    print("API 키를 설정하면 실제 Prompt Caching 효과를 확인할 수 있습니다.")
    print("\n[시뮬레이션] Prompt Caching 예상 효과:")
    print("  호출 1: 캐시 생성 (input: 350 tok, cache_creation: 350 tok, cache_read: 0)")
    print("  호출 2: 캐시 히트 (input: 30 tok, cache_creation: 0, cache_read: 350 tok) ← 90% 할인")
    print("  호출 3: 캐시 히트 (input: 25 tok, cache_creation: 0, cache_read: 350 tok) ← 90% 할인")

---
## 섹션 7. EP07 Compression + Caching 결합

EP07에서 배운 Context Compression과 Semantic Cache를 결합하면 비용 절감 효과가 극대화됩니다.

**파이프라인**: 쿼리 → Semantic Cache 확인 → (미스 시) 컨텍스트 압축 → LLM 호출 → 캐시에 저장

In [ ]:
import tiktoken

# 토큰 수 계산 유틸리티
def count_tokens(text: str, model: str = "cl100k_base") -> int:
    """tiktoken으로 토큰 수 계산"""
    enc = tiktoken.get_encoding(model)
    return len(enc.encode(text))


# 간단한 컨텍스트 압축기 (EP07 스타일)
def compress_context(text: str, query: str, compression_ratio: float = 0.5) -> str:
    """쿼리 관련 문장만 추출하는 간단한 압축기
    
    실제로는 LLMlingua나 LLMChainExtractor를 사용하지만,
    여기서는 유사도 기반 문장 필터링으로 시뮬레이션합니다.
    """
    sentences = [s.strip() for s in text.split(".") if s.strip()]
    if not sentences:
        return text
    
    # 쿼리와 각 문장의 유사도 계산
    query_emb = embedding_model.encode(query, normalize_embeddings=True)
    sent_embs = embedding_model.encode(sentences, normalize_embeddings=True)
    similarities = np.dot(sent_embs, query_emb)
    
    # 상위 문장만 선택 (compression_ratio만큼)
    n_keep = max(1, int(len(sentences) * (1 - compression_ratio)))
    top_indices = np.argsort(similarities)[-n_keep:]
    top_indices = sorted(top_indices)  # 원래 순서 유지
    
    compressed = ". ".join(sentences[i] for i in top_indices) + "."
    return compressed


# 결합 파이프라인
class CacheCompressionPipeline:
    """Semantic Cache + Context Compression 결합 파이프라인"""
    
    def __init__(self, semantic_cache, compression_ratio=0.5):
        self.cache = semantic_cache
        self.compression_ratio = compression_ratio
        self.stats = {
            "cache_hits": 0,
            "cache_misses": 0,
            "total_tokens_saved": 0,
            "total_tokens_used": 0,
        }
    
    def query(self, user_query: str, context: str) -> dict:
        """결합 파이프라인 실행"""
        result = {"query": user_query, "source": None, "response": None, "tokens": {}}
        
        # 1단계: Semantic Cache 확인
        cached = self.cache.get(user_query)
        if cached:
            self.stats["cache_hits"] += 1
            original_tokens = count_tokens(context + user_query)
            self.stats["total_tokens_saved"] += original_tokens
            result["source"] = "cache_hit"
            result["response"] = cached["response"]
            result["tokens"] = {
                "original": original_tokens,
                "used": 0,
                "saved": original_tokens
            }
            return result
        
        # 2단계: 컨텍스트 압축
        original_tokens = count_tokens(context)
        compressed = compress_context(context, user_query, self.compression_ratio)
        compressed_tokens = count_tokens(compressed)
        compression_saving = original_tokens - compressed_tokens
        
        # 3단계: LLM 호출 (시뮬레이션)
        simulated_response = f"[LLM 응답] {user_query}에 대한 답변 (압축된 컨텍스트 기반)"
        
        # 4단계: 캐시에 저장
        self.cache.put(user_query, simulated_response)
        
        self.stats["cache_misses"] += 1
        self.stats["total_tokens_used"] += compressed_tokens
        self.stats["total_tokens_saved"] += compression_saving
        
        result["source"] = "llm_with_compression"
        result["response"] = simulated_response
        result["tokens"] = {
            "original": original_tokens,
            "compressed": compressed_tokens,
            "saved": compression_saving
        }
        return result


print("CacheCompressionPipeline 정의 완료")

In [ ]:
# 결합 파이프라인 테스트

# 새로운 Semantic Cache 생성
pipeline_cache = SemanticCache(
    embedding_model=embedding_model,
    redis_client=r,
    threshold=0.92
)

pipeline = CacheCompressionPipeline(
    semantic_cache=pipeline_cache,
    compression_ratio=0.5
)

# 테스트용 컨텍스트 문서
context_doc = """당사의 환불 정책은 고객 만족을 최우선으로 합니다. 
주문 후 7일 이내에 환불을 신청할 수 있습니다. 
상품이 개봉되지 않은 상태여야 환불이 가능합니다. 
환불 처리에는 영업일 기준 3~5일이 소요됩니다. 
카드 결제의 경우 카드사에 따라 1~2주가 추가로 걸릴 수 있습니다. 
환불 금액은 원래 결제 수단으로 돌려드립니다. 
부분 환불도 가능하며, 고객센터에 문의해 주세요. 
이벤트 상품이나 할인 상품은 환불 조건이 다를 수 있습니다."""

# 쿼리 실행
queries = [
    "환불 기간이 어떻게 되나요?",       # 첫 질문: 미스 → 압축 후 LLM 호출
    "환불 절차 알려주세요",              # 유사 질문: 캐시 히트 예상
    "반품하려면 어떻게 하나요?",          # 유사 질문: 캐시 히트 예상
    "배송은 얼마나 걸리나요?",            # 다른 주제: 미스 → 압축 후 LLM 호출
]

print("=" * 70)
print("Cache + Compression 결합 파이프라인 테스트")
print("=" * 70)

for i, q in enumerate(queries):
    result = pipeline.query(q, context_doc)
    print(f"\n--- 쿼리 {i+1}: {q} ---")
    print(f"소스: {result['source']}")
    if result['source'] == 'cache_hit':
        print(f"토큰 절감: {result['tokens']['saved']} tok (LLM 호출 없음)")
    else:
        print(f"원본 토큰: {result['tokens']['original']} → 압축 후: {result['tokens']['compressed']}")
        print(f"압축으로 절감: {result['tokens']['saved']} tok")

print(f"\n{'=' * 70}")
print(f"전체 통계:")
print(f"  캐시 히트: {pipeline.stats['cache_hits']}")
print(f"  캐시 미스: {pipeline.stats['cache_misses']}")
print(f"  총 절감 토큰: {pipeline.stats['total_tokens_saved']}")
print(f"  총 사용 토큰: {pipeline.stats['total_tokens_used']}")

---
## 섹션 8. 캐시 히트율 측정 및 비용 절감 계산

실제 서비스를 시뮬레이션하여 캐시의 비용 절감 효과를 정량적으로 측정합니다.

In [ ]:
import random

# 시뮬레이션: 100개 쿼리, 30%가 유사 질문

# FAQ 기반 질문 풀 (원본 + 변형)
faq_variations = {
    "환불": [
        "환불 절차가 어떻게 되나요?", "환불하려면 어떻게 해야 하나요?",
        "반품 방법 알려주세요", "환불 가능한가요?", "환불 기간이 어떻게 되나요?"
    ],
    "배송": [
        "배송 기간은 얼마나 걸리나요?", "택배 언제 도착해요?",
        "배송은 며칠 걸려요?", "배송 조회 어떻게 하나요?"
    ],
    "탈퇴": [
        "회원 탈퇴는 어떻게 하나요?", "계정 삭제하고 싶습니다",
        "탈퇴 절차 알려주세요"
    ],
    "포인트": [
        "포인트 적립은 언제 되나요?", "적립금 확인하고 싶어요",
        "포인트 유효기간이 얼마인가요?"
    ]
}

unique_questions = [
    f"고유질문_{i}: 이것은 새로운 유형의 질문입니다 (변형 {i})" 
    for i in range(70)
]

# 100개 쿼리 생성 (30%가 FAQ 변형 = 유사 질문)
random.seed(42)
all_queries = []
for _ in range(30):  # 30% 유사 질문
    category = random.choice(list(faq_variations.keys()))
    query = random.choice(faq_variations[category])
    all_queries.append((query, "similar"))
for i in range(70):  # 70% 고유 질문
    all_queries.append((unique_questions[i], "unique"))

random.shuffle(all_queries)

print(f"총 {len(all_queries)}개 쿼리 생성")
print(f"  유사 질문: 30개 (30%)")
print(f"  고유 질문: 70개 (70%)")

In [ ]:
# 비용 절감 시뮬레이션

# 비용 파라미터 (Claude 3.5 Sonnet 기준)
COST_PER_INPUT_TOKEN = 3.0 / 1_000_000   # $3.00 / 1M tokens
COST_PER_OUTPUT_TOKEN = 15.0 / 1_000_000  # $15.00 / 1M tokens
AVG_INPUT_TOKENS = 2000   # 평균 입력 토큰
AVG_OUTPUT_TOKENS = 500   # 평균 출력 토큰

# 시뮬레이션용 Semantic Cache
sim_cache = SemanticCache(
    embedding_model=embedding_model,
    redis_client=r,
    threshold=0.92
)

# FAQ 원본을 캐시에 워밍
for category, variations in faq_variations.items():
    sim_cache.put(variations[0], f"{category}에 대한 상세 답변입니다...")

# 시뮬레이션 실행
baseline_cost = 0.0  # 캐시 없는 경우 비용
actual_cost = 0.0    # 캐시 있는 경우 비용
cache_hits = 0
cache_misses = 0

for query, query_type in all_queries:
    # 기본 비용 (캐시 없을 때)
    query_cost = (AVG_INPUT_TOKENS * COST_PER_INPUT_TOKEN + 
                  AVG_OUTPUT_TOKENS * COST_PER_OUTPUT_TOKEN)
    baseline_cost += query_cost
    
    # Semantic Cache 확인
    cached = sim_cache.get(query)
    if cached:
        cache_hits += 1
        actual_cost += 0.0001  # 캐시 조회 비용 (임베딩 계산만)
    else:
        cache_misses += 1
        actual_cost += query_cost
        # 미스된 쿼리를 캐시에 저장
        sim_cache.put(query, f"{query}에 대한 답변")

# 결과
savings = baseline_cost - actual_cost
savings_pct = savings / baseline_cost * 100

print("=" * 70)
print("비용 절감 시뮬레이션 결과 (100개 쿼리)")
print("=" * 70)
print(f"\n캐시 성능:")
print(f"  캐시 히트: {cache_hits}건")
print(f"  캐시 미스: {cache_misses}건")
print(f"  히트율: {cache_hits / len(all_queries) * 100:.1f}%")
print(f"\n비용 비교:")
print(f"  캐싱 없는 경우: ${baseline_cost:.4f}")
print(f"  Semantic Cache 적용: ${actual_cost:.4f}")
print(f"  절감액: ${savings:.4f} ({savings_pct:.1f}%)")
print(f"\n월간 예상 (일 100건 × 30일):")
print(f"  캐싱 없음: ${baseline_cost * 30:.2f}/월")
print(f"  캐싱 적용: ${actual_cost * 30:.2f}/월")
print(f"  월 절감: ${savings * 30:.2f}/월 ({savings_pct:.1f}%)")

---
## 섹션 9. 캐시 무효화 전략 구현

캐시된 응답이 오래되었거나, 데이터 소스가 변경되었을 때 자동으로 무효화하는 전략입니다.

In [ ]:
class SemanticCacheWithInvalidation(SemanticCache):
    """TTL + 버전 기반 캐시 무효화를 지원하는 Semantic Cache"""
    
    def __init__(self, embedding_model, redis_client, threshold=0.92,
                 ttl_seconds=3600, version="v1.0"):
        super().__init__(embedding_model, redis_client, threshold)
        self.ttl_seconds = ttl_seconds
        self.version = version
    
    def put(self, query: str, response: str):
        """TTL과 버전 정보를 포함하여 캐시에 저장"""
        embedding = self._get_embedding(query)
        entry = {
            "query": query,
            "embedding": embedding,
            "response": response,
            "timestamp": time.time(),
            "version": self.version,
            "expires_at": time.time() + self.ttl_seconds
        }
        self.entries.append(entry)
    
    def get(self, query: str) -> Optional[dict]:
        """TTL 만료 및 버전 불일치 항목을 제외하고 조회"""
        now = time.time()
        query_emb = self._get_embedding(query)
        
        best_match = None
        best_similarity = -1.0
        
        # 유효한 엔트리만 검색
        valid_entries = [
            e for e in self.entries
            if e["expires_at"] > now and e["version"] == self.version
        ]
        
        for entry in valid_entries:
            sim = self._cosine_similarity(query_emb, entry["embedding"])
            if sim > best_similarity:
                best_similarity = sim
                best_match = entry
        
        self.stats["similarities"].append(best_similarity)
        
        if best_match and best_similarity >= self.threshold:
            self.stats["hits"] += 1
            return {
                "response": best_match["response"],
                "similarity": best_similarity,
                "cached_query": best_match["query"],
                "version": best_match["version"],
                "ttl_remaining": best_match["expires_at"] - now
            }
        
        self.stats["misses"] += 1
        return None
    
    def invalidate_version(self, old_version: str):
        """특정 버전의 캐시 항목 무효화"""
        before = len(self.entries)
        self.entries = [e for e in self.entries if e.get("version") != old_version]
        removed = before - len(self.entries)
        return removed
    
    def cleanup_expired(self):
        """만료된 항목 정리"""
        now = time.time()
        before = len(self.entries)
        self.entries = [e for e in self.entries if e["expires_at"] > now]
        removed = before - len(self.entries)
        return removed

In [ ]:
# 캐시 무효화 시연

# TTL 2초로 설정하여 빠르게 만료 테스트
inv_cache = SemanticCacheWithInvalidation(
    embedding_model=embedding_model,
    redis_client=r,
    threshold=0.92,
    ttl_seconds=2,  # 2초 후 만료 (테스트용)
    version="v1.0"
)

# 캐시에 항목 저장
inv_cache.put("환불 절차가 어떻게 되나요?", "[v1.0] 환불은 7일 이내 가능합니다.")

# 즉시 조회 → 히트
result1 = inv_cache.get("환불하려면 어떻게 해야 하나요?")
print("=== TTL 기반 무효화 테스트 ===")
print(f"즉시 조회: {'HIT' if result1 else 'MISS'}")
if result1:
    print(f"  TTL 남은 시간: {result1['ttl_remaining']:.1f}초")

# 3초 대기 후 조회 → TTL 만료로 미스
print("\n3초 대기 중...")
time.sleep(3)
result2 = inv_cache.get("환불하려면 어떻게 해야 하나요?")
print(f"3초 후 조회: {'HIT' if result2 else 'MISS (TTL 만료)'}")

# 버전 기반 무효화 테스트
print("\n=== 버전 기반 무효화 테스트 ===")
inv_cache_v = SemanticCacheWithInvalidation(
    embedding_model=embedding_model,
    redis_client=r,
    threshold=0.92,
    ttl_seconds=3600,
    version="v1.0"
)

inv_cache_v.put("환불 절차가 어떻게 되나요?", "[v1.0] 환불은 7일 이내 가능합니다.")

# v1.0으로 조회 → 히트
result3 = inv_cache_v.get("환불하려면 어떻게 해야 하나요?")
print(f"v1.0 조회: {'HIT' if result3 else 'MISS'}")

# 버전을 v2.0으로 업데이트
inv_cache_v.version = "v2.0"
result4 = inv_cache_v.get("환불하려면 어떻게 해야 하나요?")
print(f"v2.0 조회: {'HIT' if result4 else 'MISS (버전 불일치)'}")

# 새 버전으로 캐시 갱신
inv_cache_v.put("환불 절차가 어떻게 되나요?", "[v2.0] 환불은 14일 이내 가능합니다. (정책 변경)")
result5 = inv_cache_v.get("환불하려면 어떻게 해야 하나요?")
print(f"v2.0 갱신 후 조회: {'HIT' if result5 else 'MISS'}")
if result5:
    print(f"  응답: {result5['response']}")

# v1.0 캐시 정리
removed = inv_cache_v.invalidate_version("v1.0")
print(f"\nv1.0 캐시 {removed}개 항목 무효화 완료")

---
## 섹션 10. Langfuse 통합 -- 캐시 히트/미스 추적

Langfuse를 사용하여 캐시 성능을 모니터링합니다. 각 쿼리에 `cache_hit` 또는 `cache_miss` 태그를 붙여 추적합니다.

In [ ]:
# Langfuse 클라이언트 초기화
try:
    from langfuse import Langfuse
    
    langfuse = Langfuse(
        public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
        secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
        host=os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com")
    )
    LANGFUSE_AVAILABLE = True
    print("Langfuse 클라이언트 초기화 성공")
except Exception as e:
    LANGFUSE_AVAILABLE = False
    print(f"Langfuse 연결 불가: {e}")
    print("Langfuse 키를 설정하면 실제 추적이 가능합니다.")

In [ ]:
# Langfuse 통합 캐시 쿼리 함수

def query_with_langfuse_tracking(
    query: str,
    cache: SemanticCache,
    langfuse_client=None
):
    """Langfuse 추적이 포함된 캐시 쿼리"""
    start_time = time.time()
    
    # Langfuse trace 생성
    trace = None
    if langfuse_client:
        trace = langfuse_client.trace(
            name="semantic_cache_query",
            input={"query": query}
        )
    
    # 캐시 확인
    cached = cache.get(query)
    latency = time.time() - start_time
    
    if cached:
        # 캐시 히트
        if trace:
            trace.update(
                tags=["cache_hit"],
                output={"response": cached["response"]},
                metadata={
                    "similarity": cached["similarity"],
                    "cached_query": cached["cached_query"],
                    "latency_ms": latency * 1000,
                    "estimated_cost_saved": AVG_INPUT_TOKENS * COST_PER_INPUT_TOKEN
                }
            )
            trace.generation(
                name="cache_response",
                input=query,
                output=cached["response"],
                model="semantic_cache",
                usage={"input": 0, "output": 0, "total": 0}
            )
        return {"source": "cache", "response": cached["response"], 
                "similarity": cached["similarity"], "latency_ms": latency * 1000}
    else:
        # 캐시 미스 → LLM 호출 시뮬레이션
        response = f"[LLM] {query}에 대한 새 응답입니다."
        cache.put(query, response)
        
        llm_latency = time.time() - start_time
        
        if trace:
            trace.update(
                tags=["cache_miss"],
                output={"response": response},
                metadata={
                    "latency_ms": llm_latency * 1000
                }
            )
            trace.generation(
                name="llm_response",
                input=query,
                output=response,
                model="claude-3.5-sonnet",
                usage={
                    "input": AVG_INPUT_TOKENS,
                    "output": AVG_OUTPUT_TOKENS,
                    "total": AVG_INPUT_TOKENS + AVG_OUTPUT_TOKENS
                }
            )
        return {"source": "llm", "response": response, "latency_ms": llm_latency * 1000}


# Langfuse 추적 시연
tracking_cache = SemanticCache(
    embedding_model=embedding_model,
    redis_client=r,
    threshold=0.92
)

# FAQ 워밍
for query, response in faq_data:
    tracking_cache.put(query, response)

# 테스트 쿼리
langfuse_client = langfuse if LANGFUSE_AVAILABLE else None

test_queries_tracking = [
    "환불 방법이 궁금합니다",
    "배송 언제 와요?",
    "최신 아이폰 가격이 얼마인가요?",
]

print("=" * 70)
print("Langfuse 추적이 포함된 캐시 쿼리")
print("=" * 70)

for q in test_queries_tracking:
    result = query_with_langfuse_tracking(q, tracking_cache, langfuse_client)
    tag = "cache_hit" if result["source"] == "cache" else "cache_miss"
    print(f"[{tag:>10}] {q}")
    print(f"             레이턴시: {result['latency_ms']:.2f}ms")
    if "similarity" in result:
        print(f"             유사도: {result['similarity']:.4f}")
    print()

if LANGFUSE_AVAILABLE:
    langfuse.flush()
    print("Langfuse에 trace 전송 완료 → 대시보드에서 확인하세요")
else:
    print("(Langfuse 미연결 -- 로컬 출력만 표시)")

---
## 섹션 11. Exercises

아래 두 개의 실습을 완성해 보세요.

### Exercise 1: Semantic Cache 임계값 최적화 실험

**목표**: 유사도 임계값을 변경하며 히트율과 정확도의 최적 균형점을 찾는다

**단계**:
1. 아래 코드의 `YOUR_FAQ_DATA`에 자신의 도메인 FAQ 10개를 작성
2. 각 FAQ에 대해 유사 변형 3개씩 생성 (총 30개 변형)
3. 임계값 0.85 / 0.88 / 0.90 / 0.92 / 0.95 에서 히트율과 정확도를 측정
4. 결과를 시각화하고, 최적 임계값을 선택하고 근거를 작성

In [ ]:
# Exercise 1: TODO - 임계값 최적화 실험

# TODO: 자신의 도메인 FAQ 데이터를 작성하세요
YOUR_FAQ_DATA = [
    # ("원본 질문", "정답 응답"),
    # ...
]

# TODO: 각 FAQ에 대한 유사 변형 질문을 작성하세요
YOUR_VARIATIONS = {
    # 0: ["변형1", "변형2", "변형3"],  # FAQ 인덱스: 변형 질문 리스트
    # ...
}

# TODO: 관련 없는 질문도 추가하세요 (미스가 정답인 케이스)
UNRELATED_QUERIES = [
    # "관련 없는 질문1",
    # ...
]

# TODO: 임계값별 성능 측정 코드를 작성하세요
thresholds_to_test = [0.85, 0.88, 0.90, 0.92, 0.95]
exercise_results = []

for threshold in thresholds_to_test:
    # TODO: SemanticCache를 생성하고 FAQ를 로드
    # TODO: 변형 질문 + 관련 없는 질문으로 테스트
    # TODO: 히트율, 정확도, 오답률을 계산
    pass

# TODO: 결과를 표로 출력하세요
# TODO: (보너스) matplotlib으로 임계값 vs 히트율/정확도 차트를 그리세요

print("Exercise 1: TODO를 완성하세요!")

### Exercise 2: EP07 Compression + Semantic Cache 결합 파이프라인

**목표**: Context Compression과 Semantic Cache를 결합하여 비용 절감 효과를 비교한다

**단계**:
1. 아래 `YOUR_DOCUMENTS`에 500단어 이상의 문서 2개를 작성
2. 파이프라인 A (Semantic Cache만)와 파이프라인 B (Compression + Cache)를 구현
3. 동일한 10개 쿼리로 두 파이프라인을 실행
4. 토큰 소비, 비용, 히트율을 비교 분석

In [ ]:
# Exercise 2: TODO - Compression + Cache 결합 파이프라인

# TODO: 테스트용 문서를 작성하세요 (500단어 이상)
YOUR_DOCUMENTS = [
    # "긴 문서 내용 1...",
    # "긴 문서 내용 2...",
]

# TODO: 테스트 쿼리 10개를 작성하세요 (일부는 유사 질문)
YOUR_QUERIES = [
    # "질문 1",
    # ...
]

# TODO: 파이프라인 A -- Semantic Cache만 사용
# pipeline_a = SemanticCache(...)

# TODO: 파이프라인 B -- CacheCompressionPipeline 사용
# pipeline_b = CacheCompressionPipeline(...)

# TODO: 동일 쿼리로 두 파이프라인을 실행하고 결과를 비교하세요
# 비교 항목: 총 토큰 소비, 총 비용, 캐시 히트율, 응답 품질

# TODO: (보너스) Prompt Caching까지 추가한 파이프라인 C와 비교

print("Exercise 2: TODO를 완성하세요!")

---
## 마무리

### 이번 에피소드에서 배운 것

| 개념 | 핵심 포인트 |
|------|------------|
| **Exact Cache** | 문자열 해시 비교, 간단하지만 히트율 10~25% |
| **Semantic Cache** | 임베딩 유사도 기반, 히트율 40~65% |
| **유사도 임계값** | 0.92 기본값, 도메인별 A/B 테스트로 조정 |
| **Prompt Caching** | Claude cache_control로 시스템 프롬프트 90% 할인 |
| **EP07 결합** | Compression + Caching으로 최대 84% 비용 절감 |
| **캐시 무효화** | TTL + 버전 + 의미 변화 감지 3중 전략 |
| **Langfuse 추적** | cache_hit/cache_miss 태깅으로 성능 가시화 |

### 핵심 공식

```
월 비용 절감 = 총 쿼리 수 × 캐시 히트율 × 평균 쿼리 비용
```

### 다음 에피소드: EP18. Cost Dashboard & Budget Alert

- Langfuse + Grafana 연동 비용 대시보드
- 모델별/팀별/기능별 비용 분배
- 예산 임계값 알림 (Slack/이메일)

**구독 ・ 좋아요 ・ 알림 설정** 잊지 마세요!